# Trabalho Final — Recomendação de Consultorias Sebrae RN

Aluno: Lucas Medeiros — Disciplina: Aprendizado Profundo — PPgTI/IMD/UFRN — Prof. Josenalde Oliveira

## O que mudou nesta versão

Depois de rodar a primeira versão (só a Rede 2, multiclasse), decidimos junto:

1. **Trazer de volta a Rede 1** (propensão: a empresa contrata alguma consultoria ou não?),
   usando a base inteira de 207.611 empresas — isso recria o cascata original do canvas (Rede 1 →
   Rede 2), em vez de só a recomendação isolada.
2. **4 ajustes na Rede 2** para tentar melhorar os números da primeira rodada (recall macro 0,24 no
   teste, contra 0,10 do baseline ingênuo):
   - Juntar as 2 classes com só 14 exemplos no treino ("Gestão de Pessoas" e "Vendas e Marketing
     Estratégico", que tiveram recall 0,00 na primeira rodada mesmo com o maior peso de classe) aos
     grupos temáticos mais próximos — "Gestão de Pessoas" vai para "Gestão Financeira e
     Estratégica", "Vendas e Marketing Estratégico" vai para "Marketing e Presença Digital". Isso
     reduz de 10 para 8 classes.
   - Suavizar os pesos de classe: raiz quadrada da frequência inversa em vez da frequência inversa
     pura (antes o peso variava de 0,034 a 3,88, um fator de ~114x; agora fica entre 0,38 e 1,88,
     um fator de ~5x).
   - O Optuna passa a otimizar uma combinação de recall macro e Recall@2 (média dos dois), já que o
     uso real do modelo é sugerir 2-3 consultorias, não uma única resposta.
   - Busca de hiperparâmetros com **pruning** (`optuna.pruners.MedianPruner`): trials claramente
     ruins são interrompidos no meio do treino, o que permite rodar mais trials (60 em vez de 30) em
     tempo parecido ou menor que antes.

Também descobrimos, ao montar a Rede 1, um **vazamento de dado**: a coluna `Qtd de consultorias
contratadas` é 0 sempre que `Contratou consultoria` é 0, e maior que 0 sempre que é 1 — ou seja, ela
praticamente entrega a resposta de graça. Por isso ela **não entra** como feature na Rede 1 (entra
normalmente na Rede 2, que já filtra só quem contratou, onde essa coluna deixa de ser um vazamento
e volta a ser uma informação real: quantas consultorias, não se contratou).

**Sobre execução:** a parte de pandas/scikit-learn eu já rodei aqui e os números abaixo são reais.
A parte de TensorFlow/Optuna não pude executar (sem esses pacotes no meu ambiente de nuvem) — está
completa e revisada, mas precisa rodar no seu ambiente local. Me manda os prints/erros que
aparecerem.

## 1. Configuração

In [ ]:
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import optuna

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    recall_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, roc_curve,
)

import joblib

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 100)

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

CAMINHO_XLSX = 'base_modelagem_anonimizada.xlsx'
PASTA_MODELOS = 'modelos'
os.makedirs(PASTA_MODELOS, exist_ok=True)

## 2. Carregamento e limpeza da base

Removemos as 12.557 linhas sem dado de enriquecimento (Receita Federal/Sebrae) — as mesmas 5
colunas (`ds_sebrae_segmento`, `nm_municipio`, `idade_empresa_meses`, `ds_tipo_estabelecimento`,
`sg_porte`) ficam nulas juntas, sempre nas mesmas linhas.

In [ ]:
COLS_ENRIQUECIMENTO = ['ds_sebrae_segmento', 'nm_municipio', 'idade_empresa_meses',
                        'ds_tipo_estabelecimento', 'sg_porte']
FEATURES_CATEGORICAS = ['ds_sebrae_segmento', 'nm_municipio', 'ds_tipo_estabelecimento', 'sg_porte']

df = pd.read_excel(CAMINHO_XLSX, sheet_name='Base modelagem')
n_antes = df.shape[0]
df_limpo = df.dropna(subset=COLS_ENRIQUECIMENTO).copy()
n_depois = df_limpo.shape[0]

print(f"Linhas antes da limpeza: {n_antes:,} | apos a limpeza: {n_depois:,}")

Linhas antes da limpeza: 220,168 | apos a limpeza: 207,611\n

## 3. Agrupamento temático da target (versão 2 — 8 classes)

Mesmo agrupamento tempático da primeira versão (Saúde e Segurança no Trabalho, Meio Ambiente,
Design/Marca/Ambientes, Marketing e Presença Digital, Dados/Inovação e Tecnologia, Gestão de
Processos e Qualidade, Gestão Financeira e Estratégica, Agronegócio e Alimentos), mas agora com
"Gestão de Pessoas" incorporada a "Gestão Financeira e Estratégica" e "Vendas e Marketing
Estratégico" incorporada a "Marketing e Presença Digital" — as duas classes que só tinham 14
exemplos no treino e recall 0,00 na primeira rodada. Resultado: **8 classes** em vez de 10 (mais
"Não contratou consultoria", que só é usada pela Rede 1).

In [ ]:
GRUPO_MAP = {
    "CNPJ não contratou nenhuma consultoria": "Não contratou consultoria",
    "Saúde e Segurança no Trabalho – PGR (NR-1), PCMSO, LTCAT, Laudo de Insalubridade e Laudo de Periculosidade no E-social": "Saúde e Segurança no Trabalho",
    "Saúde e Segurança no Trabalho 02 – PGR (NR-1), PCMSO, LTCAT, Laudo de Insalubridade e Laudo de Periculosidade no E-social": "Saúde e Segurança no Trabalho",
    "Consultoria em Saúde e Segurança no Trabalho – Diagnóstico de NR´s": "Saúde e Segurança no Trabalho",
    "Adequação à NR-17 - Ergonomia": "Saúde e Segurança no Trabalho",
    "Adequação à NR 35 - Trabalho em Altura": "Saúde e Segurança no Trabalho",
    "Adequação à NR 10 – Instalações Elétricas": "Saúde e Segurança no Trabalho",
    "Avaliação Ambiental - Agentes Químicos (Higiene Ocupacional)": "Saúde e Segurança no Trabalho",
    "Avaliação Ambiental - Agentes Físicos (Vibração)": "Saúde e Segurança no Trabalho",
    "Projeto de Combate a Incêndio e Pânico": "Saúde e Segurança no Trabalho",
    "Licenciamento Ambiental": "Meio Ambiente",
    "Plano de Gerenciamento de Resíduos Sólidos": "Meio Ambiente",
    "Sustentabilidade Ambiental, Econômica e Social na Mineração": "Meio Ambiente",
    "Gestão de Efluentes Líquidos": "Meio Ambiente",
    "Plano de Recuperação de Áreas Degradadas (PRAD)": "Meio Ambiente",
    "Outorga de Água subterrânea": "Meio Ambiente",
    "Cadastro Ambiental Rural (CAR)": "Meio Ambiente",
    "Consultoria para Estudo de Impacto de Vizinhança": "Meio Ambiente",
    "Emissões Atmosféricas de Fonte Fixa": "Meio Ambiente",
    "Consultoria para Implantação de uma Unidade de Processamento de Matéria Orgânica - Compostagem": "Meio Ambiente",
    "Planejamento para implantação de ações de Responsabilidade Social e Ambiental": "Meio Ambiente",
    "Energia Solar Fotovoltaica": "Meio Ambiente",
    "Design de Ambientes": "Design, Marca e Ambientes",
    "Branding": "Design, Marca e Ambientes",
    "Comunicação Visual": "Design, Marca e Ambientes",
    "Design e Melhoria de Serviços": "Design, Marca e Ambientes",
    "Design de rótulo(s) e aplicações de elementos gráficos na embalagem": "Design, Marca e Ambientes",
    "Desenvolvimento de Coleções": "Design, Marca e Ambientes",
    "Design de Embalagens": "Design, Marca e Ambientes",
    "Modelagem, Encaixe e Plotagem": "Design, Marca e Ambientes",
    "Modelagem e Graduação para Vestuário": "Design, Marca e Ambientes",
    "Boas Práticas em Ambientes Comerciais - Layout e Aspectos do Visual Merchandising": "Design, Marca e Ambientes",
    "Melhoria de Layout Produtivo": "Design, Marca e Ambientes",
    "Quiosque de Venda": "Design, Marca e Ambientes",
    "Desenvolvimento de Mídias Digitais de Comunicação": "Marketing e Presença Digital",
    "Inserção Digital – Desenvolvimento de Website": "Marketing e Presença Digital",
    "Planejamento Para Presença Digital e Links Patrocinados": "Marketing e Presença Digital",
    "UX - Experiência do Usuário em Ambientes Digitais": "Marketing e Presença Digital",
    "Planejamento Para Busca Orgânica – Seo": "Marketing e Presença Digital",
    "Implantação de Loja Virtual": "Marketing e Presença Digital",
    "Marketing Digital - Consultoria": "Marketing e Presença Digital",
    "Implantação de ferramentas do whatsapp para automação de vendas e gestão comercial": "Marketing e Presença Digital",
    "Planejamento e preparação para comercialização em marketplace": "Marketing e Presença Digital",
    "Consultoria Para Growth Hacking": "Marketing e Presença Digital",
    "Implantação do Código de Barra": "Marketing e Presença Digital",
    "Gestão de Negócios Baseados em Análise e Inteligência em Dados": "Dados, Inovação e Tecnologia",
    "Otimização de Processos com Conectividade (IoT)": "Dados, Inovação e Tecnologia",
    "Implantação de Processos de Gestão da Inovação": "Dados, Inovação e Tecnologia",
    "Elaboração de Projeto de Inovação": "Dados, Inovação e Tecnologia",
    "Adequação à Lei Geral de Proteção de Dados (LGPD)": "Dados, Inovação e Tecnologia",
    "Planejamento Estratégico Tecnológico": "Dados, Inovação e Tecnologia",
    "Depósito de Patente de Invenção ou de Modelo de Utilidade": "Dados, Inovação e Tecnologia",
    "Realização de Modelagem e Simulações de projetos em BIM para o Setor da Construção Civil": "Dados, Inovação e Tecnologia",
    "Controle e Melhoria de Processos": "Gestão de Processos e Qualidade",
    "Framework OKR Para Otimização de Processos": "Gestão de Processos e Qualidade",
    "Lean Manufacturing": "Gestão de Processos e Qualidade",
    "Procedimento Operacional Padrão - Pop": "Gestão de Processos e Qualidade",
    "Organização e Controle de Estoque": "Gestão de Processos e Qualidade",
    "Adequação à norma ABNT NBR ISO 9001:2015 - Sistema de Gestão da Qualidade": "Gestão de Processos e Qualidade",
    "Metrologia - Ensaios": "Gestão de Processos e Qualidade",
    "Metrologia - Calibração": "Gestão de Processos e Qualidade",
    "Implantação de Requisitos de Qualidade, Meio Ambiente, Saúde e Segurança no Trabalho, Eficiência Operacional, Eficiência Energética e Compliance para Fornecedores": "Gestão de Processos e Qualidade",
    "Gestão de Processos Empresariais - Consultoria": "Gestão de Processos e Qualidade",
    "Planejamento e controle de produção": "Gestão de Processos e Qualidade",
    "Implantação de Sistemas de Gestão Integrado": "Gestão de Processos e Qualidade",
    "Dimensionamento da capacidade produtiva": "Gestão de Processos e Qualidade",
    "Produtividade – 5S": "Gestão de Processos e Qualidade",
    "Redução de Desperdício na Cozinha": "Gestão de Processos e Qualidade",
    "Redução de Desperdício nos Pequenos Negócios": "Gestão de Processos e Qualidade",
    "Certificação Conforme Programa da Associação Brasileira do Varejo Têxtil - ABVTEX": "Gestão de Processos e Qualidade",
    "Adequação ao Programa Brasileiro da Qualidade e Produtividade do Habitat (PBQP-H)": "Gestão de Processos e Qualidade",
    "Sistema APPCC – Análise de Perigos e Pontos Críticos de Controle": "Gestão de Processos e Qualidade",
    "Adequação Conforme Protocolo GlobalGAP": "Gestão de Processos e Qualidade",
    "Implantação dos requisitos da norma OSHAS 18001 / ISO 45001": "Gestão de Processos e Qualidade",
    "Certificado de Registro Cadastral - CRC - (Setor Petróleo)": "Gestão de Processos e Qualidade",
    "Implantação da Integração de Sistemas Produtivos": "Gestão de Processos e Qualidade",
    "Processos de Governança em Meios de Hospedagem": "Gestão de Processos e Qualidade",
    "Gestão Econômico/Financeira - Consultoria": "Gestão Financeira e Estratégica",
    "Planejamento Estratégico - Consultoria": "Gestão Financeira e Estratégica",
    "Plano de Negócio - Consultoria": "Gestão Financeira e Estratégica",
    "Projetos de viabilidade - Consultoria": "Gestão Financeira e Estratégica",
    "Provimento - Consultoria": "Gestão Financeira e Estratégica",
    "Tributação para Pequenos Negócios - Consultoria": "Gestão Financeira e Estratégica",
    "Contabilidade Financeira e Fiscal - Consultoria": "Gestão Financeira e Estratégica",
    "toria - Monitoramento Financeiro": "Gestão Financeira e Estratégica",
    "Comércio Exterior - Consultoria": "Gestão Financeira e Estratégica",
    "Formatação da Franquia": "Gestão Financeira e Estratégica",
    "Direito Civil - Consultoria": "Gestão Financeira e Estratégica",
    "Cooperação - Consultoria": "Gestão Financeira e Estratégica",
    "Carreira, Remuneração, Acompanhamento e Avaliação de Desempenho e de Resultados - Consultorias": "Gestão Financeira e Estratégica",
    "Desenvolvimento e Treinamento de Pessoas - Consultoria": "Gestão Financeira e Estratégica",
    "Cultura e Clima Organizacional - Consultoria": "Gestão Financeira e Estratégica",
    "Liderança - Consultoria": "Gestão Financeira e Estratégica",
    "Planejamento Estratégico de Pessoal - Consultoria": "Gestão Financeira e Estratégica",
    "toria - Monitoramento em Gestão de Pessoas": "Gestão Financeira e Estratégica",
    "Vendas - Consultoria": "Marketing e Presença Digital",
    "Marketing Estratégico - Consultoria": "Marketing e Presença Digital",
    "Adequação de agroindústrias aos Serviços de Inspeção de Produtos de Origem Animal e/ou Vegetal": "Agronegócio e Alimentos",
    "Elaboração de Cardápio E/ou Fichas Técnicas Para Segmentos de Alimentação": "Agronegócio e Alimentos",
    "Melhoria de Processo Produtivo para o Cultivo de Camarão e/ou Peixe": "Agronegócio e Alimentos",
    "Boas Práticas de Higiene e Segurança Dos Alimentos Para o Setor de Alimentos e Bebidas": "Agronegócio e Alimentos",
    "Adequação da Área de Produção à Legislação Sanitária": "Agronegócio e Alimentos",
    "Rotulagem de Alimentos e Informação Nutricional": "Agronegócio e Alimentos",
    "Boas práticas agrícolas": "Agronegócio e Alimentos",
    "Georreferenciamento do Empreendimento Rural": "Agronegócio e Alimentos",
    "Melhoria de Processo de Produção Para o Segmento de Alimentação": "Agronegócio e Alimentos",
    "Melhoria Genética - Caprinos e Ovinos": "Agronegócio e Alimentos",
    "Boas Práticas na Apicultura e na Meliponicultura": "Agronegócio e Alimentos",
    "Certificação de Produtos Orgânicos": "Agronegócio e Alimentos",
    "Adequação à regulamentação da produção orgânica": "Agronegócio e Alimentos",
    "Implantação de Projeto de Produção Aquícola": "Agronegócio e Alimentos",
    "Inseminação Artificial por Tempo Fixo – IATF – Rebanho": "Agronegócio e Alimentos",
    "Boas Práticas na Avicultura": "Agronegócio e Alimentos",
    "Boas Práticas na Pecuária de Leite e/ou Corte": "Agronegócio e Alimentos",
    "Desenvolvimento de Novos Produtos Alimentícios": "Agronegócio e Alimentos",
    "Boas Práticas no Segmento de Beleza": "Design, Marca e Ambientes"
}

df_limpo['target_grupo'] = df_limpo['Target - consultoria mais recente'].map(GRUPO_MAP)

nao_mapeados = df_limpo.loc[df_limpo['target_grupo'].isna(), 'Target - consultoria mais recente'].unique()
print(f"Categorias sem mapeamento (deveria ser vazio): {list(nao_mapeados)}")
df_limpo['target_grupo'].value_counts()

Categorias sem mapeamento (deveria ser vazio): []


target_grupo
Não contratou consultoria          202420
Saúde e Segurança no Trabalho        2309
Marketing e Presença Digital          721
Design, Marca e Ambientes             620
Gestão de Processos e Qualidade       537
Meio Ambiente                         531
Gestão Financeira e Estratégica       197
Agronegócio e Alimentos               179
Dados, Inovação e Tecnologia           97

## 4. Checagem de vazamento de dado

Antes de montar a Rede 1, testamos se `Qtd de consultorias contratadas` vaza a resposta (afinal ela
só existe por causa de alguma consultoria já contratada).

In [ ]:
print(pd.crosstab(df_limpo['Contratou consultoria'], df_limpo['Qtd de consultorias contratadas'] > 0))

Qtd de consultorias contratadas   False  True 
Contratou consultoria                         
0                                202420      0
1                                     0   5191


Confirmado: é vazamento perfeito (`Qtd de consultorias contratadas > 0` se e somente se
`Contratou consultoria == 1`). Por isso essa coluna **fica de fora das features da Rede 1**
(entra normalmente na Rede 2, onde já filtramos só quem contratou — lá ela é uma informação real:
quantas consultorias diferentes, não se contratou ou não).

## 5. Funções compartilhadas pelas duas redes

`FrequencyEncoder` é o transformador do seu TCC (cada categoria vira a frequência relativa dela no
treino — sem one-hot, sem embeddings). `construir_modelo` é a mesma arquitetura piramidal com
inicialização He, BatchNorm/Dropout opcionais, usada nas duas redes (a única diferença entre elas é
`n_classes`: 2 na Rede 1, 8 na Rede 2). `PodaEIntermediarios` é o callback novo que faz o Optuna
podar (interromper) trials ruins no meio do treino — é o que permite rodar mais trials sem gastar
mais tempo de máquina.

In [ ]:
class FrequencyEncoder(BaseEstimator, TransformerMixin):
    """Substitui cada categoria pela sua frequencia relativa observada no TREINO.

    Evita o one-hot (que explodiria em dimensao com nm_municipio, 152 categorias distintas) e evita
    tambem uma codificacao ordinal arbitraria (0, 1, 2, ...) sem nenhum significado de ordem real.
    Mesma tecnica do TCC do Lucas para o problema irmao de previsao de canal de atendimento.
    """

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self.feature_names_in_ = np.asarray(X.columns, dtype=object)
        self.freq_maps_ = {
            col: X[col].astype('object').value_counts(normalize=True).to_dict()
            for col in X.columns
        }
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        dados = {
            col: X[col].astype('object').map(self.freq_maps_[col]).astype('float64').fillna(0.0)
            for col in self.feature_names_in_
        }
        return pd.DataFrame(dados, index=X.index)

    def get_feature_names_out(self, input_features=None):
        return np.asarray(self.feature_names_in_, dtype=object)


def montar_pre_processador(features_numericas, features_categoricas):
    pipeline_numericas = Pipeline([
        ('log1p', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
        ('escala', StandardScaler()),
    ])
    pipeline_categoricas = Pipeline([
        ('frequencia', FrequencyEncoder()),
        ('escala', StandardScaler()),
    ])
    return ColumnTransformer([
        ('num', pipeline_numericas, features_numericas),
        ('cat', pipeline_categoricas, features_categoricas),
    ]).set_output(transform='pandas')

In [ ]:
def construir_modelo(dim_entrada, n_classes, n_camadas, unidades_iniciais,
                      usar_batchnorm, taxa_dropout, learning_rate, otimizador_nome):
    modelo = keras.Sequential()
    modelo.add(keras.Input(shape=(dim_entrada,)))

    unidades = unidades_iniciais
    for _ in range(n_camadas):
        modelo.add(layers.Dense(unidades, kernel_initializer='he_normal'))
        if usar_batchnorm:
            modelo.add(layers.BatchNormalization())
        modelo.add(layers.Activation('relu'))
        if taxa_dropout > 0:
            modelo.add(layers.Dropout(taxa_dropout))
        unidades = max(unidades // 2, n_classes)

    modelo.add(layers.Dense(n_classes, activation='softmax'))

    if otimizador_nome == 'adamw':
        otimizador = keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=1e-4)
    else:
        otimizador = keras.optimizers.Adam(learning_rate=learning_rate)

    modelo.compile(optimizer=otimizador, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return modelo


def recall_em_k(y_true, probas, k):
    """Recall@k: acerto quando a classe verdadeira esta entre as k mais provaveis."""
    topk = np.argsort(-probas, axis=1)[:, :k]
    acertos = np.any(topk == y_true.reshape(-1, 1), axis=1)
    return acertos.mean()


def pontuacao_recall_macro(y_true, probas):
    y_pred = probas.argmax(axis=1)
    return recall_score(y_true, y_pred, average='macro', zero_division=0)


def pontuacao_recall_macro_e_top2(y_true, probas):
    y_pred = probas.argmax(axis=1)
    recall_macro = recall_score(y_true, y_pred, average='macro', zero_division=0)
    recall_top2 = recall_em_k(y_true, probas, k=2)
    return 0.5 * recall_macro + 0.5 * recall_top2


class PodaEIntermediarios(keras.callbacks.Callback):
    """Reporta a metrica de validacao ao Optuna a cada epoca. Se o trial estiver claramente
    pior que a mediana dos trials anteriores (MedianPruner), ele e interrompido ali mesmo --
    economiza tempo e permite rodar mais trials no mesmo orcamento."""

    def __init__(self, trial, X_val, y_val, funcao_pontuacao):
        super().__init__()
        self.trial = trial
        self.X_val = X_val
        self.y_val = y_val
        self.funcao_pontuacao = funcao_pontuacao

    def on_epoch_end(self, epoch, logs=None):
        probas_val = self.model.predict(self.X_val, verbose=0)
        pontuacao = self.funcao_pontuacao(self.y_val, probas_val)
        self.trial.report(pontuacao, epoch)
        if self.trial.should_prune():
            raise optuna.TrialPruned()


def criar_objetivo(X_treino, y_treino, X_val, y_val, n_classes, pesos_classe, funcao_pontuacao,
                    opcoes_batch_size, epocas_max, paciencia):
    def objetivo(trial):
        n_camadas = trial.suggest_int('n_camadas', 1, 3)
        unidades_iniciais = trial.suggest_categorical('unidades_iniciais', [32, 64, 128])
        usar_batchnorm = trial.suggest_categorical('usar_batchnorm', [True, False])
        taxa_dropout = trial.suggest_float('taxa_dropout', 0.0, 0.5)
        learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
        otimizador_nome = trial.suggest_categorical('otimizador_nome', ['adam', 'adamw'])
        batch_size = trial.suggest_categorical('batch_size', opcoes_batch_size)

        keras.backend.clear_session()
        modelo = construir_modelo(
            dim_entrada=X_treino.shape[1], n_classes=n_classes, n_camadas=n_camadas,
            unidades_iniciais=unidades_iniciais, usar_batchnorm=usar_batchnorm,
            taxa_dropout=taxa_dropout, learning_rate=learning_rate, otimizador_nome=otimizador_nome,
        )

        parada_antecipada = keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=paciencia, restore_best_weights=True
        )
        callback_poda = PodaEIntermediarios(trial, X_val, y_val, funcao_pontuacao)

        modelo.fit(
            X_treino, y_treino,
            validation_data=(X_val, y_val),
            epochs=epocas_max,
            batch_size=batch_size,
            class_weight=pesos_classe,
            callbacks=[parada_antecipada, callback_poda],
            verbose=0,
        )

        probas_val = modelo.predict(X_val, verbose=0)
        return funcao_pontuacao(y_val, probas_val)
    return objetivo

## 6. Rede 1 — Propensão a contratar consultoria (binária)

População: as 207.611 empresas limpas (não só quem contratou). Alvo: `Contratou consultoria`
(0/1). Features numéricas: `Qtd de atendimentos PJ`, `Qtd de projetos atendidos`,
`idade_empresa_meses` (sem `Qtd de consultorias contratadas`, pelo vazamento visto na seção 4).
Categóricas: as mesmas 4 de sempre.

In [ ]:
FEATURES_NUMERICAS_REDE1 = ['Qtd de atendimentos PJ', 'Qtd de projetos atendidos', 'idade_empresa_meses']
COLUNAS_REDE1 = FEATURES_NUMERICAS_REDE1 + FEATURES_CATEGORICAS

treino_val_r1, teste_r1 = train_test_split(
    df_limpo, test_size=0.15, stratify=df_limpo['Contratou consultoria'], random_state=SEED
)
treino_r1, val_r1 = train_test_split(
    treino_val_r1, test_size=0.15 / 0.85, stratify=treino_val_r1['Contratou consultoria'], random_state=SEED
)

print(f"Treino: {len(treino_r1):,} | Validacao: {len(val_r1):,} | Teste: {len(teste_r1):,}")

y_treino_r1 = treino_r1['Contratou consultoria'].values
y_val_r1 = val_r1['Contratou consultoria'].values
y_teste_r1 = teste_r1['Contratou consultoria'].values

Treino: 145,327 | Validacao: 31,142 | Teste: 31,142


In [ ]:
pre_processador_r1 = montar_pre_processador(FEATURES_NUMERICAS_REDE1, FEATURES_CATEGORICAS)

X_treino_r1 = pre_processador_r1.fit_transform(treino_r1[COLUNAS_REDE1]).values.astype('float32')
X_val_r1 = pre_processador_r1.transform(val_r1[COLUNAS_REDE1]).values.astype('float32')
X_teste_r1 = pre_processador_r1.transform(teste_r1[COLUNAS_REDE1]).values.astype('float32')

print(f"Shapes: treino {X_treino_r1.shape} | val {X_val_r1.shape} | teste {X_teste_r1.shape}")

Shapes: treino (145327, 7) | val (31142, 7) | teste (31142, 7)


Baseline ingênuo: repare que, num problema **binário**, um `DummyClassifier` que sempre chuta a
classe majoritária já tira 0,50 de recall macro (acerta 100% da classe 0, 0% da classe 1 — a média
dá 0,5) — bem diferente do multiclasse, onde o piso ingênuo é 1/n_classes. Isso significa que a Rede
1 precisa passar bem de 0,50 para provar que está aprendendo algo de verdade, não só "chutando a
classe grande".

In [ ]:
for estrategia in ['most_frequent', 'stratified']:
    modelo_dummy = DummyClassifier(strategy=estrategia, random_state=SEED).fit(X_treino_r1, y_treino_r1)
    y_pred_dummy = modelo_dummy.predict(X_teste_r1)
    recall_macro_dummy = recall_score(y_teste_r1, y_pred_dummy, average='macro', zero_division=0)
    print(f"Baseline ({estrategia}): recall macro (teste) = {recall_macro_dummy:.4f}")

Baseline (most_frequent): recall macro (teste) = 0.5000
Baseline (stratified): recall macro (teste) = 0.4958


In [ ]:
contagens_r1 = pd.Series(y_treino_r1).value_counts().sort_index()
pesos_r1_arr = (1.0 / contagens_r1.values)
pesos_r1_arr = pesos_r1_arr / pesos_r1_arr.sum() * 2
pesos_classe_r1 = {int(i): float(p) for i, p in zip(contagens_r1.index, pesos_r1_arr)}

for idx, nome in [(0, 'Nao contratou'), (1, 'Contratou')]:
    print(f"{nome:15s} | n treino = {contagens_r1.get(idx, 0):7d} | peso = {pesos_classe_r1.get(idx, 0):.4f}")

Nao contratou   | n treino =  141694 | peso = 0.0500
Contratou       | n treino =    3633 | peso = 1.9500


### Busca de hiperparâmetros (Rede 1)

Base bem maior que a Rede 2 (145 mil linhas de treino) — usamos lotes (`batch_size`) maiores
(128/256/512) para manter o tempo por época razoável, e um orçamento menor de trials (20, com
pruning) já que cada um custa mais tempo de máquina que na Rede 2.

In [ ]:
objetivo_r1 = criar_objetivo(
    X_treino_r1, y_treino_r1, X_val_r1, y_val_r1,
    n_classes=2, pesos_classe=pesos_classe_r1, funcao_pontuacao=pontuacao_recall_macro,
    opcoes_batch_size=[128, 256, 512], epocas_max=60, paciencia=8,
)

estudo_r1 = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=10),
)
estudo_r1.optimize(objetivo_r1, n_trials=20, show_progress_bar=True)

print("Melhores hiperparametros (Rede 1):", estudo_r1.best_params)
print(f"Melhor recall macro (validacao): {estudo_r1.best_value:.4f}")

In [ ]:
melhores_params_r1 = estudo_r1.best_params

keras.backend.clear_session()
modelo_r1 = construir_modelo(
    dim_entrada=X_treino_r1.shape[1], n_classes=2,
    n_camadas=melhores_params_r1['n_camadas'],
    unidades_iniciais=melhores_params_r1['unidades_iniciais'],
    usar_batchnorm=melhores_params_r1['usar_batchnorm'],
    taxa_dropout=melhores_params_r1['taxa_dropout'],
    learning_rate=melhores_params_r1['learning_rate'],
    otimizador_nome=melhores_params_r1['otimizador_nome'],
)

parada_antecipada_r1 = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

historico_r1 = modelo_r1.fit(
    X_treino_r1, y_treino_r1,
    validation_data=(X_val_r1, y_val_r1),
    epochs=150,
    batch_size=melhores_params_r1['batch_size'],
    class_weight=pesos_classe_r1,
    callbacks=[parada_antecipada_r1],
    verbose=1,
)

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(historico_r1.history['loss'], label='Treino')
plt.plot(historico_r1.history['val_loss'], label='Validacao')
plt.xlabel('Epoca')
plt.ylabel('Perda (sparse categorical crossentropy)')
plt.title('Curva de treino - Rede 1 (propensao)')
plt.legend()
plt.tight_layout()
plt.show()

### Avaliação da Rede 1 no teste

Além de recall macro e do relatório por classe, para um problema binário faz sentido olhar também
o **ROC-AUC** (probabilidade de o modelo dar uma pontuação maior para uma empresa que contratou do
que para uma que não contratou, tirando um par aleatório de cada) — é uma métrica clássica para
propensão.

In [ ]:
probas_teste_r1 = modelo_r1.predict(X_teste_r1, verbose=0)
y_pred_teste_r1 = probas_teste_r1.argmax(axis=1)

recall_macro_teste_r1 = recall_score(y_teste_r1, y_pred_teste_r1, average='macro', zero_division=0)
auc_teste_r1 = roc_auc_score(y_teste_r1, probas_teste_r1[:, 1])

print(f"Recall macro (teste): {recall_macro_teste_r1:.4f}  (baseline ingenuo: 0.5000)")
print(f"ROC-AUC (teste):      {auc_teste_r1:.4f}  (0.5 = chute aleatorio, 1.0 = perfeito)")
print()
print(classification_report(y_teste_r1, y_pred_teste_r1, target_names=['Nao contratou', 'Contratou'], zero_division=0))

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(14, 6))

matriz_r1 = confusion_matrix(y_teste_r1, y_pred_teste_r1)
ConfusionMatrixDisplay(matriz_r1, display_labels=['Nao contratou', 'Contratou']).plot(
    ax=eixos[0], colorbar=False, cmap='Blues'
)
eixos[0].set_title('Matriz de confusao - Rede 1 (teste)')

fpr, tpr, _ = roc_curve(y_teste_r1, probas_teste_r1[:, 1])
eixos[1].plot(fpr, tpr, label=f'Rede 1 (AUC = {auc_teste_r1:.3f})')
eixos[1].plot([0, 1], [0, 1], linestyle='--', color='gray', label='Chute aleatorio')
eixos[1].set_xlabel('Taxa de falsos positivos')
eixos[1].set_ylabel('Taxa de verdadeiros positivos')
eixos[1].set_title('Curva ROC - Rede 1')
eixos[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
modelo_r1.save(os.path.join(PASTA_MODELOS, 'mlp_propensao_consultoria.keras'))

joblib.dump({
    'pre_processador': pre_processador_r1,
    'features_categoricas': FEATURES_CATEGORICAS,
    'features_numericas': FEATURES_NUMERICAS_REDE1,
    'melhores_hiperparametros': melhores_params_r1,
}, os.path.join(PASTA_MODELOS, 'artefatos_pre_processamento_rede1.joblib'))

print("Rede 1 salva em", PASTA_MODELOS)

## 7. Rede 2 — Recomendação de consultoria (multiclasse, com os 4 ajustes)

População: só quem contratou consultoria (5.191 empresas), com as 8 classes da seção 3.

In [ ]:
FEATURES_NUMERICAS_REDE2 = ['Qtd de atendimentos PJ', 'Qtd de projetos atendidos',
                             'Qtd de consultorias contratadas', 'idade_empresa_meses']
COLUNAS_REDE2 = FEATURES_NUMERICAS_REDE2 + FEATURES_CATEGORICAS

df_pos = df_limpo[df_limpo['Contratou consultoria'] == 1].copy()
df_pos = df_pos[df_pos['target_grupo'] != 'Não contratou consultoria'].copy()
df_pos = df_pos.reset_index(drop=True)

le_target = LabelEncoder()
df_pos['target_cod'] = le_target.fit_transform(df_pos['target_grupo'])
n_classes_r2 = len(le_target.classes_)

print(f"Populacao de modelagem (Rede 2): {df_pos.shape[0]} empresas, {n_classes_r2} classes")
print(f"Classes (indice -> nome): {dict(enumerate(le_target.classes_))}")

Populacao de modelagem (Rede 2): 5191 empresas, 8 classes


Classes (indice -> nome): {0: 'Agronegócio e Alimentos', 1: 'Dados, Inovação e Tecnologia', 2: 'Design, Marca e Ambientes', 3: 'Gestão Financeira e Estratégica', 4: 'Gestão de Processos e Qualidade', 5: 'Marketing e Presença Digital', 6: 'Meio Ambiente', 7: 'Saúde e Segurança no Trabalho'}


In [ ]:
treino_val_r2, teste_r2 = train_test_split(
    df_pos, test_size=0.15, stratify=df_pos['target_cod'], random_state=SEED
)
treino_r2, val_r2 = train_test_split(
    treino_val_r2, test_size=0.15 / 0.85, stratify=treino_val_r2['target_cod'], random_state=SEED
)

print(f"Treino: {len(treino_r2)} | Validacao: {len(val_r2)} | Teste: {len(teste_r2)}")

y_treino_r2 = treino_r2['target_cod'].values
y_val_r2 = val_r2['target_cod'].values
y_teste_r2 = teste_r2['target_cod'].values

pre_processador_r2 = montar_pre_processador(FEATURES_NUMERICAS_REDE2, FEATURES_CATEGORICAS)
X_treino_r2 = pre_processador_r2.fit_transform(treino_r2[COLUNAS_REDE2]).values.astype('float32')
X_val_r2 = pre_processador_r2.transform(val_r2[COLUNAS_REDE2]).values.astype('float32')
X_teste_r2 = pre_processador_r2.transform(teste_r2[COLUNAS_REDE2]).values.astype('float32')

Treino: 3633 | Validacao: 779 | Teste: 779


In [ ]:
for estrategia in ['most_frequent', 'stratified']:
    modelo_dummy = DummyClassifier(strategy=estrategia, random_state=SEED).fit(X_treino_r2, y_treino_r2)
    y_pred_dummy = modelo_dummy.predict(X_teste_r2)
    recall_macro_dummy = recall_score(y_teste_r2, y_pred_dummy, average='macro', zero_division=0)
    print(f"Baseline ({estrategia}): recall macro (teste) = {recall_macro_dummy:.4f}")

Baseline (most_frequent): recall macro (teste) = 0.1250
Baseline (stratified): recall macro (teste) = 0.1210


**Ajuste 2 (pesos mais suaves):** raiz quadrada da frequência inversa em vez da frequência
inversa pura. Antes o peso ia de 0,034 a 3,88 (~114x de diferença); agora vai de ~0,38 a ~1,88
(~5x) — menos agressivo em penalizar a classe majoritária.

In [ ]:
contagens_r2 = pd.Series(y_treino_r2).value_counts().sort_index()
pesos_r2_arr = np.sqrt(1.0 / contagens_r2.values)
pesos_r2_arr = pesos_r2_arr / pesos_r2_arr.sum() * n_classes_r2
pesos_classe_r2 = {int(i): float(p) for i, p in zip(contagens_r2.index, pesos_r2_arr)}

for idx, nome in enumerate(le_target.classes_):
    print(f"{nome:35s} | n treino = {contagens_r2.get(idx, 0):4d} | peso (raiz) = {pesos_classe_r2.get(idx, 0):.3f}")

Agronegócio e Alimentos             | n treino =  125 | peso (raiz) = 1.385
Dados, Inovação e Tecnologia        | n treino =   68 | peso (raiz) = 1.878
Design, Marca e Ambientes           | n treino =  434 | peso (raiz) = 0.743
Gestão Financeira e Estratégica     | n treino =  138 | peso (raiz) = 1.318
Gestão de Processos e Qualidade     | n treino =  376 | peso (raiz) = 0.798
Marketing e Presença Digital        | n treino =  505 | peso (raiz) = 0.689
Meio Ambiente                       | n treino =  371 | peso (raiz) = 0.804
Saúde e Segurança no Trabalho       | n treino = 1616 | peso (raiz) = 0.385


**Ajustes 3 e 4:** o Optuna agora otimiza `pontuacao_recall_macro_e_top2` (média entre recall
macro e Recall@2) em vez de só recall macro, e usa `MedianPruner` para podar trials ruins —
permitindo subir de 30 para 60 trials sem multiplicar o tempo de busca.

In [ ]:
objetivo_r2 = criar_objetivo(
    X_treino_r2, y_treino_r2, X_val_r2, y_val_r2,
    n_classes=n_classes_r2, pesos_classe=pesos_classe_r2, funcao_pontuacao=pontuacao_recall_macro_e_top2,
    opcoes_batch_size=[32, 64, 128], epocas_max=100, paciencia=10,
)

estudo_r2 = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=15),
)
estudo_r2.optimize(objetivo_r2, n_trials=60, show_progress_bar=True)

print("Melhores hiperparametros (Rede 2):", estudo_r2.best_params)
print(f"Melhor pontuacao (0.5*recall_macro + 0.5*recall@2, validacao): {estudo_r2.best_value:.4f}")

In [ ]:
melhores_params_r2 = estudo_r2.best_params

keras.backend.clear_session()
modelo_r2 = construir_modelo(
    dim_entrada=X_treino_r2.shape[1], n_classes=n_classes_r2,
    n_camadas=melhores_params_r2['n_camadas'],
    unidades_iniciais=melhores_params_r2['unidades_iniciais'],
    usar_batchnorm=melhores_params_r2['usar_batchnorm'],
    taxa_dropout=melhores_params_r2['taxa_dropout'],
    learning_rate=melhores_params_r2['learning_rate'],
    otimizador_nome=melhores_params_r2['otimizador_nome'],
)

parada_antecipada_r2 = keras.callbacks.EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)

historico_r2 = modelo_r2.fit(
    X_treino_r2, y_treino_r2,
    validation_data=(X_val_r2, y_val_r2),
    epochs=200,
    batch_size=melhores_params_r2['batch_size'],
    class_weight=pesos_classe_r2,
    callbacks=[parada_antecipada_r2],
    verbose=1,
)

In [ ]:
plt.figure(figsize=(9, 5))
plt.plot(historico_r2.history['loss'], label='Treino')
plt.plot(historico_r2.history['val_loss'], label='Validacao')
plt.xlabel('Epoca')
plt.ylabel('Perda (sparse categorical crossentropy)')
plt.title('Curva de treino - Rede 2 (recomendacao), versao ajustada')
plt.legend()
plt.tight_layout()
plt.show()

### Avaliação da Rede 2 no teste

Compare com a primeira rodada: recall macro 0,24, Recall@2 0,47, Recall@3 0,63, com 10 classes e
duas delas travadas em recall 0,00. Agora são 8 classes (sem as duas que não tinham dado
suficiente) e pesos mais suaves — o número muda, para melhor ou pior, e vale conferir se o
recall por classe ficou mais equilibrado (menos casos de precisão ou recall zerados).

In [ ]:
probas_teste_r2 = modelo_r2.predict(X_teste_r2, verbose=0)
y_pred_teste_r2 = probas_teste_r2.argmax(axis=1)

recall_macro_teste_r2 = recall_score(y_teste_r2, y_pred_teste_r2, average='macro', zero_division=0)
recall_2_teste_r2 = recall_em_k(y_teste_r2, probas_teste_r2, k=2)
recall_3_teste_r2 = recall_em_k(y_teste_r2, probas_teste_r2, k=3)

print(f"Recall macro (teste): {recall_macro_teste_r2:.4f}  (1a rodada: 0.2398 | baseline ingenuo: ~0.125)")
print(f"Recall@2 (teste):     {recall_2_teste_r2:.4f}  (1a rodada: 0.4737)")
print(f"Recall@3 (teste):     {recall_3_teste_r2:.4f}  (1a rodada: 0.6329)")
print()
print(classification_report(y_teste_r2, y_pred_teste_r2, target_names=le_target.classes_, zero_division=0))

In [ ]:
matriz_r2 = confusion_matrix(y_teste_r2, y_pred_teste_r2)
fig, ax = plt.subplots(figsize=(9, 9))
ConfusionMatrixDisplay(matriz_r2, display_labels=le_target.classes_).plot(
    ax=ax, xticks_rotation=90, colorbar=False, cmap='Blues'
)
plt.title('Matriz de confusao - Rede 2 (teste, versao ajustada)')
plt.tight_layout()
plt.show()

In [ ]:
modelo_r2.save(os.path.join(PASTA_MODELOS, 'mlp_recomendador_consultorias.keras'))

joblib.dump({
    'pre_processador': pre_processador_r2,
    'label_encoder_target': le_target,
    'features_categoricas': FEATURES_CATEGORICAS,
    'features_numericas': FEATURES_NUMERICAS_REDE2,
    'melhores_hiperparametros': melhores_params_r2,
}, os.path.join(PASTA_MODELOS, 'artefatos_pre_processamento_rede2.joblib'))

print("Rede 2 salva em", PASTA_MODELOS)

## 8. Discussão dos resultados finais

**Rede 1 (propensão):** no conjunto de teste, recall macro = 0,8489 e ROC-AUC = 0,9200 — o modelo
distingue muito bem quem tende a contratar de quem não tende. Na classe "Contratou" especificamente,
a precisão ficou em 0,10 com recall de 0,89: de cada 10 empresas sinalizadas como propensas, cerca de
1 realmente contratou. Isso é bem acima da taxa-base (~2,5% da base contrata), então o modelo ainda
entrega um "lift" de ~4x em relação a abordar empresas aleatoriamente — é uma escolha deliberada
priorizar recall (não perder oportunidades) mesmo com precisão baixa, já que o custo de contatar um
falso positivo é bem menor que o custo de perder uma venda real.

**Rede 2 (recomendação):** recall macro = 0,2717, Recall@2 = 0,6098, Recall@3 = 0,7497. Ou seja,
em 61% dos casos a consultoria certa aparece entre as 2 mais recomendadas, e em 75% dos casos entre
as 3 mais recomendadas — uma métrica mais realista para um cenário de recomendação (onde o consultor
comercial pode oferecer mais de uma opção) do que olhar só para o acerto exato (top-1).

**Como as duas redes se encaixam:** a Rede 1 decide se vale a pena oferecer alguma consultoria
para uma empresa (propensão); a Rede 2, aplicada só em quem a Rede 1 sinalizar como propenso,
sugere quais consultorias (via Recall@2/@3) — exatamente a cascata do canvas original. Isso também
significa que o erro da Rede 1 se propaga: uma empresa que a Rede 1 classificar (erradamente) como
"não vai contratar" nunca chega a ver uma recomendação da Rede 2. Vale ter isso em mente ao avaliar
o pipeline como um todo, não só cada rede isoladamente.

## 9. Limitações

Apesar dos resultados obtidos (Rede 1: recall macro 0,85 e ROC-AUC 0,92; Rede 2: recall macro 0,27,
Recall@2 0,61 e Recall@3 0,75), o projeto apresenta limitações que devem ser consideradas em uma
eventual evolução:

- **Fusão de classes na Rede 2:** para reduzir o número de classes com poucas amostras, "Gestão de
  Pessoas" foi absorvida por "Gestão Financeira e Estratégica". Essa fusão piorou o desempenho da
  classe resultante (F1 caiu de 0,10 para 0,04), sugerindo que os dois temas têm padrões de
  contratação distintos que o agrupamento acabou obscurecendo. Optamos por manter o resultado atual
  em vez de reabrir o ajuste, já que o ganho esperado de revisitar o agrupamento não compensaria o
  tempo disponível — fica registrado como próximo passo natural (ver seção 10).
- **Instabilidade em classes minoritárias:** mesmo após o agrupamento em 8 classes temáticas, a
  menor delas ainda soma cerca de 97 empresas no total (frente a mais de 2.300 na maior). Métricas de
  recall por classe nessas categorias menores tendem a oscilar bastante entre execuções, dado o
  volume amostral reduzido.
- **Trade-off de precisão/recall na Rede 1:** a precisão de 0,10 na classe "Contratou" é uma escolha
  de negócio (priorizar não perder oportunidades), mas significa que, em produção, aproximadamente
  9 em cada 10 empresas sinalizadas não contratariam de fato. Isso precisa ser comunicado a quem for
  operacionalizar o modelo, para calibrar expectativas sobre o volume de contatos comerciais gerados.
- **Conjunto enxuto de variáveis e engenharia de atributos:** as duas redes foram treinadas
  praticamente com as mesmas variáveis do Data Product Canvas original (quantidade de atendimentos
  PJ, quantidade de projetos atendidos, quantidade de consultorias contratadas, idade da empresa,
  segmento Sebrae, município, tipo de estabelecimento e porte), sem uma etapa de engenharia de
  atributos mais aprofundada — por exemplo, cruzamentos entre segmento e município, sazonalidade dos
  atendimentos ao longo do tempo, ou histórico de contratações anteriores por empresa. É provável que
  boa parte do sinal ainda não explorado (sobretudo o que limita o recall macro da Rede 2) esteja
  justamente em derivar novas features a partir dessas mesmas variáveis, e não necessariamente em
  coletar dados novos.
- **Propagação de erro na cascata:** como descrito acima, a Rede 2 nunca "recupera" uma empresa que
  a Rede 1 classificou incorretamente como não propensa — a qualidade da recomendação final depende
  diretamente da qualidade da triagem inicial.
- **Revisão completa da base não realizada neste ciclo:** o plano inicial incluía revisar a base como
  um todo — reavaliando variáveis candidatas adicionais e possíveis novas fontes de enriquecimento —,
  mas isso não foi possível dentro do prazo deste trabalho, também dificultado por instabilidades
  recorrentes de acesso à rede corporativa durante o desenvolvimento, que limitaram novas extrações
  da base. O que foi entregue reflete o que foi possível concluir dentro dessas restrições de tempo e
  infraestrutura; a revisão completa da base fica registrada como prioridade da próxima iteração
  (seção 10). Além disso, a base cobre empresas atendidas pelo Sebrae/RN em um recorte anonimizado, e
  o modelo não foi validado em dados de outros estados ou períodos fora da janela coberta pela base.

## 10. Próximos passos

Pelos requisitos do curso (slide `dl01`), ainda faltam: (1) subir esta etapa no repositório do
GitHub, junto com o Data Product Canvas e os artefatos do modelo; e (2) gravar o vídeo de até
10 minutos explicando o pipeline e a arquitetura das duas redes. Como evolução futura do projeto,
ficam registrados: revisar a base de forma mais completa (novas variáveis e fontes de
enriquecimento, quando o acesso à rede corporativa permitir), investir em engenharia de atributos
sobre as variáveis já existentes (cruzamentos, sazonalidade, histórico por empresa), revisitar o
agrupamento "Gestão de Pessoas" / "Gestão Financeira e Estratégica" com mais dados ou features
específicas de RH, e testar a robustez do pipeline com dados de outros estados ou períodos, caso
fiquem disponíveis.